In [1]:
import pandas as pd
from sqlalchemy import create_engine

In [2]:
username = "root"
password = "15042026"
host = "localhost"
database = "formula1_analytics"

engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}/{database}"
)

print("✅ MySQL connection created!")

✅ MySQL connection created!


In [3]:
with engine.connect() as connection:
    print("✅ Successfully connected to MySQL!")

✅ Successfully connected to MySQL!


In [4]:
base_path = "../data/cleaned/"

drivers = pd.read_csv(base_path + "drivers.csv")
constructors = pd.read_csv(base_path + "constructors.csv")
circuits = pd.read_csv(base_path + "circuits.csv")
status = pd.read_csv(base_path + "status.csv")
seasons = pd.read_csv(base_path + "seasons.csv")
races = pd.read_csv(base_path + "races.csv")
qualifying = pd.read_csv(base_path + "qualifying.csv")
results = pd.read_csv(base_path + "results.csv")
driver_standings = pd.read_csv(base_path + "driver_standings.csv")
constructor_standings = pd.read_csv(base_path + "constructor_standings.csv")
constructor_results = pd.read_csv(base_path + "constructor_results.csv")
sprint_results = pd.read_csv(base_path + "sprint_results.csv")
lap_times = pd.read_csv(base_path + "lap_times.csv")
pit_stops = pd.read_csv(base_path + "pit_stops.csv")

print("✅ All CSV files loaded!")

✅ All CSV files loaded!


In [5]:
dataframes = {
    "drivers": drivers,
    "constructors": constructors,
    "circuits": circuits,
    "status": status,
    "seasons": seasons,
    "races": races,
    "qualifying": qualifying,
    "results": results,
    "driver_standings": driver_standings,
    "constructor_standings": constructor_standings,
    "constructor_results": constructor_results,
    "sprint_results": sprint_results,
    "lap_times": lap_times,
    "pit_stops": pit_stops
}

for df in dataframes.values():
    df.replace(r"\N", pd.NA, inplace=True)

print("✅ Missing-value cleanup complete!")

✅ Missing-value cleanup complete!


In [6]:
import_order = [
    "seasons",
    "status",
    "constructors",
    "circuits",
    "drivers",
    "races",
    "qualifying",
    "results",
    "driver_standings",
    "constructor_standings",
    "constructor_results",
    "sprint_results",
    "lap_times",
    "pit_stops"
]

In [7]:
def import_table(table_name):

    df = dataframes[table_name]

    print(f"Importing {table_name}...")

    df.to_sql(
        name=table_name,
        con=engine,
        if_exists="append",
        index=False
    )

    print(f"✅ {table_name} imported successfully!")

In [8]:
import_table("seasons")

Importing seasons...
✅ seasons imported successfully!


In [9]:
for table_name in import_order[1:]:

    try:
        import_table(table_name)

    except Exception as e:
        print(f"❌ Error importing {table_name}")
        print(e)
        break

Importing status...
✅ status imported successfully!
Importing constructors...
✅ constructors imported successfully!
Importing circuits...
✅ circuits imported successfully!
Importing drivers...
✅ drivers imported successfully!
Importing races...
✅ races imported successfully!
Importing qualifying...
✅ qualifying imported successfully!
Importing results...
✅ results imported successfully!
Importing driver_standings...
✅ driver_standings imported successfully!
Importing constructor_standings...
✅ constructor_standings imported successfully!
Importing constructor_results...
✅ constructor_results imported successfully!
Importing sprint_results...
✅ sprint_results imported successfully!
Importing lap_times...
❌ Error importing lap_times
(pymysql.err.IntegrityError) (1062, "Duplicate entry '372-77-3' for key 'lap_times.PRIMARY'")
[SQL: INSERT INTO lap_times (`raceId`, `driverId`, lap, position, time, milliseconds) VALUES (%(raceId)s, %(driverId)s, %(lap)s, %(position)s, %(time)s, %(millisecon

In [10]:
lap_times[
    lap_times.duplicated(
        subset=["raceId", "driverId", "lap"],
        keep=False
    )
].sort_values(
    ["raceId", "driverId", "lap"]
).head(20)

,raceId,driverId,lap,position,time,milliseconds
101984,356,65,3,8,1:36.617,96617
231493,356,65,3,8,1:36.616,96616
101985,356,65,4,7,1:36.802,96802
231494,356,65,4,7,1:36.801,96801
101987,356,65,6,7,1:36.653,96653
231495,356,65,6,7,1:36.652,96652
101996,356,65,15,8,1:36.022,96022
231496,356,65,15,8,1:36.021,96021
101999,356,65,18,6,1:36.510,96510
231497,356,65,18,6,1:36.509,96509


In [11]:
duplicates = lap_times[
    lap_times.duplicated(
        subset=["raceId", "driverId", "lap"],
        keep=False
    )
]

print("Duplicate rows:", len(duplicates))
print("Duplicate keys:", duplicates[
    ["raceId", "driverId", "lap"]
].drop_duplicates().shape[0])

Duplicate rows: 544
Duplicate keys: 272


In [12]:
duplicate_keys = (
    lap_times[
        lap_times.duplicated(
            subset=["raceId", "driverId", "lap"],
            keep=False
        )
    ]
    .groupby(["raceId", "driverId", "lap"])
    .size()
)

print(duplicate_keys.value_counts())

2    272
Name: count, dtype: int64


In [13]:
lap_times = lap_times.drop_duplicates(
    subset=["raceId", "driverId", "lap"],
    keep="first"
).reset_index(drop=True)

print("Rows after cleaning:", len(lap_times))

Rows after cleaning: 876525


In [14]:
lap_times.to_csv(
    "../data/cleaned/lap_times.csv",
    index=False
)

print("✅ Cleaned lap_times.csv saved!")

✅ Cleaned lap_times.csv saved!


In [15]:
lap_times = pd.read_csv(
    "../data/cleaned/lap_times.csv"
)

print(lap_times.shape)

(876525, 6)


In [16]:
import_table("lap_times")

Importing lap_times...


IntegrityError: (pymysql.err.IntegrityError) (1062, "Duplicate entry '479-137-1' for key 'lap_times.PRIMARY'")
[SQL: INSERT INTO lap_times (`raceId`, `driverId`, lap, position, time, milliseconds) VALUES (%(raceId)s, %(driverId)s, %(lap)s, %(position)s, %(time)s, %(milliseconds)s)]
[parameters: [{'raceId': 479, 'driverId': 137, 'lap': 1, 'position': 1, 'time': '1:42.085', 'milliseconds': 102085}, {'raceId': 479, 'driverId': 137, 'lap': 2, 'position': 2, 'time': '1:36.287', 'milliseconds': 96287}, {'raceId': 479, 'driverId': 137, 'lap': 3, 'position': 2, 'time': '1:34.627', 'milliseconds': 94627}, {'raceId': 479, 'driverId': 137, 'lap': 4, 'position': 2, 'time': '1:34.041', 'milliseconds': 94041}, {'raceId': 479, 'driverId': 137, 'lap': 5, 'position': 2, 'time': '1:33.699', 'milliseconds': 93699}, {'raceId': 479, 'driverId': 137, 'lap': 6, 'position': 2, 'time': '1:34.244', 'milliseconds': 94244}, {'raceId': 479, 'driverId': 137, 'lap': 7, 'position': 2, 'time': '1:34.002', 'millisecond

In [17]:
lap_times = pd.read_csv("../data/cleaned/lap_times.csv")

dataframes["lap_times"] = lap_times

print(lap_times.shape)
print(dataframes["lap_times"].shape)

(876525, 6)
(876525, 6)


In [18]:
print(
    lap_times.duplicated(
        subset=["raceId", "driverId", "lap"]
    ).sum()
)

0


In [19]:
import_table("lap_times")

Importing lap_times...
✅ lap_times imported successfully!


In [20]:
pit_stops.duplicated(
    subset=["raceId", "driverId", "stop"]
).sum()

np.int64(0)

In [21]:
import_table("pit_stops")

Importing pit_stops...
✅ pit_stops imported successfully!
